# Assignment 2: Advanced RAG Techniques
## Day 6 Session 2 - Advanced RAG Fundamentals

**OBJECTIVE:** Implement advanced RAG techniques including postprocessors, response synthesizers, and structured outputs.

**LEARNING GOALS:**
- Understand and implement node postprocessors for filtering and reranking
- Learn different response synthesis strategies (TreeSummarize, Refine)
- Create structured outputs using Pydantic models
- Build advanced retrieval pipelines with multiple processing stages

**DATASET:** Use the same data folder as Assignment 1 (`Day_6/session_2/data/`)

**PREREQUISITES:** Complete Assignment 1 first

**INSTRUCTIONS:**
1. Complete each function by replacing the TODO comments with actual implementation
2. Run each cell after completing the function to test it
3. The answers can be found in the `03_advanced_rag_techniques.ipynb` notebook
4. Each technique builds on the previous one


In [1]:
# Running locally - skipping Google Drive mount
print("Running in local environment")

Running in local environment


In [2]:
%pip install -r "C:\Users\devan\ai-accelerator-C7\requirements.txt"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'C:\\Users\\devan\\ai-accelerator-C7\\requirements.txt'


In [3]:
import os
from getpass import getpass

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter key")
print("✓ OpenRouter key set successfully")

✓ OpenRouter key set successfully


In [4]:
import os
from pathlib import Path
from typing import Dict, List, Optional, Any
from pydantic import BaseModel, Field

# Core LlamaIndex components
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, Settings
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import VectorIndexRetriever

# Vector store
from llama_index.vector_stores.lancedb import LanceDBVectorStore

# Embeddings and LLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openrouter import OpenRouter

# Advanced RAG components (we'll use these in the assignments)
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.response_synthesizers import TreeSummarize, Refine, CompactAndRefine
from llama_index.core.output_parsers import PydanticOutputParser

print("Advanced RAG libraries imported successfully!")


c:\Users\devan\ai-accelerator-C7\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Advanced RAG libraries imported successfully!


In [5]:
# Configure Advanced RAG Settings (Using OpenRouter)
def setup_advanced_rag_settings():
    """
    Configure LlamaIndex with optimized settings for advanced RAG.
    Uses local embeddings and OpenRouter for LLM operations.
    """
    # Check for OpenRouter API key
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        print("⚠️ OPENROUTER_API_KEY not found in environment variables")
        print("Please ensure you have set the API key in the previous cell or in your environment.")
    else:
        print("✅ OPENROUTER_API_KEY found - full advanced RAG functionality available")

        # Configure OpenRouter LLM
        Settings.llm = OpenRouter(
            api_key=api_key,
            model="gpt-4o-mini",
            temperature=0.1  # Lower temperature for more consistent responses
        )

    # Configure local embeddings (no API key required)
    Settings.embed_model = HuggingFaceEmbedding(
        model_name="BAAI/bge-small-en-v1.5",
        trust_remote_code=True
    )

    # Advanced RAG configuration
    Settings.chunk_size = 512  # Smaller chunks for better precision
    Settings.chunk_overlap = 50

    print("Advanced RAG settings configured")
    print("- Chunk size: 512 (optimized for precision)")
    print("- Using local embeddings for cost efficiency")
    if api_key:
        print("- OpenRouter LLM ready for response synthesis")

# Setup the configuration
setup_advanced_rag_settings()


✅ OPENROUTER_API_KEY found - full advanced RAG functionality available


The Transformer `cache_dir` argument is deprecated. Please pass `cache_dir` via `model_kwargs`, `processor_kwargs`, and/or `config_kwargs` instead.
c:\Users\devan\ai-accelerator-C7\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\devan\AppData\Local\llama_index\llama_index\Cache\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-starte

Advanced RAG settings configured
- Chunk size: 512 (optimized for precision)
- Using local embeddings for cost efficiency
- OpenRouter LLM ready for response synthesis


In [7]:
# Setup: Create index from Assignment 1 (reuse the basic functionality)
def setup_basic_index(data_folder: str, force_rebuild: bool = False):
    """
    Create a basic vector index that we'll enhance with advanced techniques.
    This reuses the concepts from Assignment 1.
    """
    # Create vector store
    vector_store = LanceDBVectorStore(
        uri="./storage/advanced_rag_vectordb",
        table_name="documents"
    )

    # Load documents
    if not Path(data_folder).exists():
        print(f"Data folder not found: {data_folder}")
        return None

    reader = SimpleDirectoryReader(input_dir=data_folder, recursive=True)
    documents = reader.load_data()

    # Create storage context and index
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True
    )

    print(f"Basic index created with {len(documents)} documents")
    print("Ready for advanced RAG techniques!")
    return index

data_folder_path = "../data"
# Create the basic index
print("Setting up basic index for advanced RAG...")
index = setup_basic_index(data_folder=data_folder_path)

if index:
    print("Ready to implement advanced RAG techniques!")
else:
    print("Failed to create index - check data folder path")


Table documents doesn't exist yet. Please add some data to create it.


Setting up basic index for advanced RAG...


Generating embeddings: 100%|██████████| 1666/1666 [02:29<00:00, 11.18it/s]


Basic index created with 21 documents
Ready for advanced RAG techniques!
Ready to implement advanced RAG techniques!


## 1. Node Postprocessors - Similarity Filtering

**Concept:** Postprocessors refine retrieval results after the initial vector search. The `SimilarityPostprocessor` filters out chunks that fall below a relevance threshold.

**Why it matters:** Raw vector search often returns some irrelevant results. Filtering improves precision and response quality.

Complete the function below to create a query engine with similarity filtering.


In [8]:
from llama_index.core.postprocessor import SimilarityPostprocessor

In [9]:
def create_query_engine_with_similarity_filter(index, similarity_cutoff: float = 0.3, top_k: int = 10):
    """
    Create a query engine that filters results based on similarity scores.

    TODO: Complete this function to create a query engine with similarity postprocessing.

    Args:
        index: Vector index to query
        similarity_cutoff: Minimum similarity score (0.0 to 1.0)
        top_k: Number of initial results to retrieve before filtering

    Returns:
        Query engine with similarity filtering
    """
    # HINT: Use index.as_query_engine() with node_postprocessors parameter containing SimilarityPostprocessor

    # TODO: Create similarity postprocessor with the cutoff threshold
    # similarity_processor = ?
    similarity_processor = SimilarityPostprocessor(
        similarity_cutoff=similarity_cutoff
    )

    # TODO: Create query engine with similarity filtering
    # query_engine = ?
    query_engine = index.as_query_engine(
        similarity_top_k=top_k,
        node_postprocessors=[similarity_processor]
    )

    # return query_engine
    return query_engine

# Test the function
if index:
    filtered_engine = create_query_engine_with_similarity_filter(index, similarity_cutoff=0.3)

    if filtered_engine:
        print("Query engine with similarity filtering created")

        # Test query
        test_query = "What are the benefits of AI agents?"
        print(f"\nTesting query: '{test_query}'")

        # Uncomment when implemented:
        response = filtered_engine.query(test_query)
        print(f"Response: {response}")
        print(f"Filtered Response: {response}")
    else:
        print("Failed to create filtered query engine")
else:
    print("No index available - run previous cells first")

Query engine with similarity filtering created

Testing query: 'What are the benefits of AI agents?'
Response: The benefits of AI agents include their ability to automate tasks, enhance decision-making processes, improve efficiency, and provide personalized experiences. They can analyze large datasets quickly, adapt to changing environments, and operate continuously without fatigue. Additionally, AI agents can assist in complex problem-solving and facilitate better communication and interaction in various applications.
Filtered Response: The benefits of AI agents include their ability to automate tasks, enhance decision-making processes, improve efficiency, and provide personalized experiences. They can analyze large datasets quickly, adapt to changing environments, and operate continuously without fatigue. Additionally, AI agents can assist in complex problem-solving and facilitate better communication and interaction in various applications.


## 2. Structured Outputs with Pydantic Models

**Concept:** Structured outputs ensure predictable, parseable responses using Pydantic models. This is essential for API endpoints and data pipelines.

**Why it matters:** Instead of free-text responses, you get type-safe, validated data structures that applications can reliably process.

Complete the function below to create a structured output system for extracting research paper information.


In [17]:
from pydantic import BaseModel, Field
from typing import List

class ResearchPaperInfo(BaseModel):
    """Structured information about a research paper or AI concept."""
    title: str = Field(description="The main title or concept name")
    key_points: List[str] = Field(description="3-5 main points or findings")
    applications: List[str] = Field(description="Practical applications or use cases")
    summary: str = Field(description="Brief 2-3 sentence summary")

from llama_index.core.program import LLMTextCompletionProgram

def create_structured_output_program(output_model: BaseModel = ResearchPaperInfo):
    """
    Create a structured output program using Pydantic models.

    TODO: Complete this function to create a structured output program.

    Args:
        output_model: Pydantic model class for structured output

    Returns:
        LLMTextCompletionProgram that returns structured data
    """

    # TODO: Create output parser with the Pydantic model (HINT: Use PydanticOutputParser)
    # output_parser = ?
    output_parser = PydanticOutputParser(output_model)

    # TODO: Create the structured output program (HINT output_parser and prompt_template_str will be passed to the function)
    # program = LLMTextCompletionProgram.from_defaults(?)

    prompt_template_str = (
        "Extract structured research or AI concept information from the following context:\n"
        "{context}\n\n"
        "Question: {query}\n\n"
        "Return the answer using the requested structured output format."
    )

    program = LLMTextCompletionProgram.from_defaults(
        output_parser=output_parser,
        prompt_template_str=prompt_template_str,
        verbose=True
    )

    # return program
    return program

structured_program = create_structured_output_program(
    ResearchPaperInfo
)

print(structured_program)

if index:
    structured_program = create_structured_output_program(
        ResearchPaperInfo
    )

    if structured_program:
        print("Structured output program created")

        structure_query = "Tell me about AI agents and their capabilities"

        print(f"Testing structured query: '{structure_query}'")

        retriever = VectorIndexRetriever(
            index=index,
            similarity_top_k=3
        )

        nodes = retriever.retrieve(structure_query)

        context = "\n".join(
            [node.text for node in nodes]
        )

        response = structured_program(
            context=context,
            query=structure_query
        )

        print("\nStructured Response:")
        print(response)

    else:
        print("Failed to create structured output program")
else:
    print("No index available - run previous cells first")
    

Structured output program created
Testing structured query: 'Tell me about AI agents and their capabilities'

Structured Response:
title='AI Agents and Their Capabilities' key_points=['AI agents can autonomously perform tasks and make decisions based on data inputs.', 'They utilize machine learning algorithms to improve their performance over time.', 'AI agents can interact with users and other systems through natural language processing.', 'They are capable of learning from their environment and adapting to new situations.', 'AI agents can be deployed in various domains, including healthcare, finance, and customer service.'] applications=['Automated customer support through chatbots.', 'Predictive analytics in finance for risk assessment.', 'Personalized recommendations in e-commerce.', 'Robotic process automation in manufacturing.', 'Smart assistants for home automation.'] summary='AI agents are intelligent systems designed to perform tasks autonomously, leveraging machine learning a